# MLP-VaR Softplus SiLU regularizado - 2010-2024, w=1

Nuevo intento a partir de `Softplus + dynamic features + SiLU LayerNorm` sin modificar la funcion de perdida ni el target.

Cambios frente al SiLU dinamico anterior:

1. `silu_dropout020`: misma red `128-64`, pero `Dropout(0.20)`.
2. `silu_small`: red mas pequena `64-32` con `Dropout(0.10)`.
3. `silu_selected_features`: misma red `128-64`, `Dropout(0.10)`, pero solo conserva las features dinamicas mas directas.

La perdida sigue siendo exactamente pinball sobre `target` en escala original con `w=1`.

In [1]:
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import chi2
from torch.utils.data import DataLoader, TensorDataset

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable, *args, **kwargs):
        return iterable

torch.set_num_threads(1)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

c:\Users\GONZA\projects\TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATASET_PATH = DATA_DIR / "dataset_tfg.pkl"

ALPHA = 0.99
SIG = 0.05
W_LOSS = 1
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42
EVAL_START = "2010-01-01"
EVAL_END = "2024-12-31"

EXPERIMENT_PREFIX = "softplus_silu_regularized_2010_2024_w1"
EXPERIMENTS = [
    {
        "key": "silu_dropout020",
        "label": "Softplus + dynamic SiLU LayerNorm dropout 0.20",
        "architecture": "silu_layernorm_dropout020",
        "feature_set": "full",
        "prefix": "softplus_silu_dropout020_2010_2024_w1",
        "pred_file": "nn_softplus_silu_dropout020_2010_2024_w1.csv",
    },
    {
        "key": "silu_small",
        "label": "Softplus + dynamic SiLU LayerNorm small",
        "architecture": "silu_layernorm_small",
        "feature_set": "full",
        "prefix": "softplus_silu_small_2010_2024_w1",
        "pred_file": "nn_softplus_silu_small_2010_2024_w1.csv",
    },
    {
        "key": "silu_selected_features",
        "label": "Softplus + SiLU LayerNorm selected features",
        "architecture": "silu_layernorm",
        "feature_set": "selected",
        "prefix": "softplus_silu_selected_features_2010_2024_w1",
        "pred_file": "nn_softplus_silu_selected_features_2010_2024_w1.csv",
    },
]

## Carga de datos y features dinamicas

In [3]:
dataset = pd.read_pickle(DATASET_PATH).copy().sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El indice debe ser DatetimeIndex"
assert "target" in dataset.columns, "Falta target"
assert "rp_lag_0" in dataset.columns, "Falta rp_lag_0"
assert "vol_20" in dataset.columns, "Falta vol_20"
assert "vol_60" in dataset.columns, "Falta vol_60"

base_feature_cols = [c for c in dataset.columns if c != "target"]

# Senales del modelo dinamico anterior: ratios de volatilidad, shocks y regimen.
eps = 1e-8
abs_r = dataset["rp_lag_0"].abs()
vol_5 = dataset["rp_lag_0"].rolling(5).std(ddof=0)
vol_10 = dataset["rp_lag_0"].rolling(10).std(ddof=0)
vol_20_realized = dataset["rp_lag_0"].rolling(20).std(ddof=0)
vol_20_ref = dataset["vol_20"].abs()
vol_60_ref = dataset["vol_60"].abs()

# Base del mejor MLP anterior.
dataset["vol_5_ratio"] = vol_5 / (vol_20_realized + eps)
dataset["vol_10_ratio"] = vol_10 / (vol_20_realized + eps)
dataset["shock_1"] = abs_r / (vol_20_realized + eps)
dataset["shock_5"] = abs_r.rolling(5).max() / (vol_20_realized + eps)

# Features dinamicas del modelo SiLU anterior.
dataset["vol_20_ratio_60"] = vol_20_ref / (vol_60_ref + eps)
dataset["vol_20_ratio_252"] = vol_20_ref / (vol_20_ref.rolling(252).median() + eps)
dataset["abs_ret_sum_5_ratio"] = abs_r.rolling(5).sum() / (vol_20_realized + eps)
dataset["abs_ret_sum_10_ratio"] = abs_r.rolling(10).sum() / (vol_20_realized + eps)
dataset["max_abs_ret_10_ratio"] = abs_r.rolling(10).max() / (vol_20_realized + eps)

dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()

dynamic_features_full = [
    "vol_5_ratio", "vol_10_ratio", "shock_1", "shock_5",
    "vol_20_ratio_60", "vol_20_ratio_252",
    "abs_ret_sum_5_ratio", "abs_ret_sum_10_ratio", "max_abs_ret_10_ratio",
]
dynamic_features_selected = [
    "vol_5_ratio", "vol_10_ratio", "shock_1", "shock_5",
    "vol_20_ratio_60", "vol_20_ratio_252",
]

FEATURE_SETS = {
    "full": base_feature_cols + dynamic_features_full,
    "selected": base_feature_cols + dynamic_features_selected,
}

print("Dataset:", dataset.shape)
print("Rango:", dataset.index.min().date(), "->", dataset.index.max().date())
print("Features full:", len(FEATURE_SETS["full"]))
print("Features selected:", len(FEATURE_SETS["selected"]))
print("Features dinamicas full:", [c for c in dynamic_features_full if c in dataset.columns])
print("Features dinamicas selected:", [c for c in dynamic_features_selected if c in dataset.columns])
print("target mean/std/min/max:", dataset["target"].mean(), dataset["target"].std(), dataset["target"].min(), dataset["target"].max())

assert len(FEATURE_SETS["full"]) == 31, f"Se esperaban 31 features full, hay {len(FEATURE_SETS['full'])}"
assert len(FEATURE_SETS["selected"]) == 28, f"Se esperaban 28 features selected, hay {len(FEATURE_SETS['selected'])}"
for feature_set in FEATURE_SETS.values():
    assert all(c in dataset.columns for c in feature_set), "Faltan features en dataset"
assert dataset["target"].abs().quantile(0.999) < 0.2, "El target parece tener escala inesperada"

Dataset: (4680, 32)
Rango: 2006-04-05 -> 2024-12-27
Features full: 31
Features selected: 28
Features dinamicas full: ['vol_5_ratio', 'vol_10_ratio', 'shock_1', 'shock_5', 'vol_20_ratio_60', 'vol_20_ratio_252', 'abs_ret_sum_5_ratio', 'abs_ret_sum_10_ratio', 'max_abs_ret_10_ratio']
Features dinamicas selected: ['vol_5_ratio', 'vol_10_ratio', 'shock_1', 'shock_5', 'vol_20_ratio_60', 'vol_20_ratio_252']
target mean/std/min/max: -0.00017689210141721946 0.00751750450881661 -0.07361457194412012 0.0784301570334365


## Metricas de backtesting

In [4]:
def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc = -2 * np.log(((1 - p) ** (T - x) * p ** x) / ((1 - p_hat) ** (T - x) * p_hat ** x))
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or np.isnan(LRind):
        return {"LRind": LRind, "p_ind": p_ind, "LRcc": np.nan, "p_cc": np.nan}
    LRcc = LRuc + LRind
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": 1 - chi2.cdf(LRcc, df=2)}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y = hit[lags:]
    X = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ = float((beta.T @ XtX @ beta) / sigma2)
    return {"DQ": DQ, "p_dq": 1 - chi2.cdf(DQ, df=X.shape[1])}

In [5]:
def describe_scale(name, x):
    x = np.asarray(x, dtype=float)
    print(f"\n{name}")
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))
    print("mean:", np.nanmean(x))
    print("p95:", np.nanpercentile(x, 95))
    print("p99:", np.nanpercentile(x, 99))
    print("p99.9:", np.nanpercentile(x, 99.9))


def plausibility_report(df, var_col="VaR_pred", loss_col="loss_real"):
    describe_scale("loss_real", df[loss_col].values)
    describe_scale(var_col, df[var_col].values)
    n_negative = int((df[var_col] < 0).sum())
    max_date = df[var_col].idxmax()
    min_date = df[var_col].idxmin()
    print("n_negative_var:", n_negative)
    print("max VaR:", max_date, float(df.loc[max_date, var_col]))
    print("min VaR:", min_date, float(df.loc[min_date, var_col]))


def evaluate_var_model(df, model_label, alpha=0.99, sig=0.05):
    real_losses = df["loss_real"].values
    var_preds = df["VaR_pred"].values
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    violations = int(I.sum())
    violation_rate = violations / T
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    df_2020 = df.loc["2020"] if (df.index.year == 2020).any() else pd.DataFrame(columns=df.columns)
    row = {
        "Model": model_label,
        "w": W_LOSS,
        "T": T,
        "Expected Viol.": (1 - alpha) * T,
        "Violations": violations,
        "Violation Rate": violation_rate,
        "VR Gap": abs(violation_rate - (1 - alpha)),
        "Coverage Bias": violation_rate - (1 - alpha),
        "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
        "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var_preds)),
        "Max VaR Level": float(np.nanmax(var_preds)),
        "Min VaR Level": float(np.nanmin(var_preds)),
        "Mean VaR 2020": float(df_2020["VaR_pred"].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020["VaR_pred"].max()) if len(df_2020) else np.nan,
        "n_negative_var": int((df["VaR_pred"] < 0).sum()),
        "p_UC": kup["p_uc"],
        "UC Test": "PASS" if kup["p_uc"] > sig else "FAIL",
        "p_CC": cc["p_cc"],
        "CC Test": "PASS" if cc["p_cc"] > sig else "FAIL",
        "p_DQ": dq["p_dq"],
        "DQ Test": "PASS" if dq["p_dq"] > sig else "FAIL",
    }
    vals = [row["p_UC"], row["p_CC"], row["p_DQ"]]
    row["p_Mean"] = float(np.mean([v for v in vals if pd.notnull(v)]))
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return pd.DataFrame([row])


def evaluate_by_year(df, model_label, alpha=0.99):
    rows = []
    for year, g in df.groupby("year"):
        real_losses = g["loss_real"].values
        var_preds = g["VaR_pred"].values
        violations = int(g["viol"].sum())
        expected = (1 - alpha) * len(g)
        kup = kupiec_test(real_losses, var_preds, alpha=alpha)
        cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
        dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
        rows.append({
            "Model": model_label,
            "year": int(year),
            "T": len(g),
            "Expected Viol.": expected,
            "Violations": violations,
            "Violation Rate": violations / len(g),
            "VR Gap": abs((violations / len(g)) - (1 - alpha)),
            "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
            "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
            "Mean VaR Level": float(np.nanmean(var_preds)),
            "Max VaR Level": float(np.nanmax(var_preds)),
            "Min VaR Level": float(np.nanmin(var_preds)),
            "Max Loss": float(np.nanmax(real_losses)),
            "Mean Loss": float(np.nanmean(real_losses)),
            "n_negative_var": int((g["VaR_pred"] < 0).sum()),
            "p_UC": kup["p_uc"],
            "UC Test": "PASS" if kup["p_uc"] > SIG else "FAIL",
            "p_CC": cc["p_cc"],
            "CC Test": "PASS" if cc["p_cc"] > SIG else "FAIL",
            "p_DQ": dq["p_dq"],
            "DQ Test": "PASS" if dq["p_dq"] > SIG else "FAIL",
        })
    return pd.DataFrame(rows)


def violation_table(df):
    cols = ["VaR_pred", "loss_real", "viol", "year"]
    out = df.loc[df["viol"] == 1, cols].copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["exceedance_ratio"] = out["loss_real"] / out["VaR_pred"]
    return out.sort_index()


def worst_days_table(df, n=25):
    out = df.copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["loss_minus_var"] = out["exceedance"]
    return out.sort_values("loss_real", ascending=False).head(n)


def violation_gap_summary(df):
    v = df.loc[df["viol"] == 1].copy()
    if len(v) < 2:
        return {"violations": len(v), "median_gap_days": np.nan, "gaps_le_5": 0, "gaps_le_20": 0}
    gaps = pd.Series(v.index).diff().dt.days.dropna()
    return {
        "violations": len(v),
        "median_gap_days": float(gaps.median()),
        "gaps_le_5": int((gaps <= 5).sum()),
        "gaps_le_20": int((gaps <= 20).sum()),
    }

## Modelos y entrenamiento

In [6]:
def inverse_softplus(y):
    return math.log(math.expm1(y))


# IMPORTANTE: no modificar. Misma perdida pinball que los experimentos Softplus anteriores.
def var_loss(y_true, y_pred, q=0.99, w=1.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, torch.as_tensor(w, dtype=y_pred.dtype, device=y_pred.device), torch.ones_like(e))
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplusSiLULayerNorm(nn.Module):
    def __init__(self, d_in, hidden=(128, 64), dropout=0.10):
        super().__init__()
        h1, h2 = hidden
        self.body = nn.Sequential(
            nn.Linear(d_in, h1),
            nn.LayerNorm(h1),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.LayerNorm(h2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(h2, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


def build_model(d_in, architecture):
    if architecture == "silu_layernorm":
        return MLPVaRSoftplusSiLULayerNorm(d_in=d_in, hidden=(128, 64), dropout=0.10)
    if architecture == "silu_layernorm_dropout020":
        return MLPVaRSoftplusSiLULayerNorm(d_in=d_in, hidden=(128, 64), dropout=0.20)
    if architecture == "silu_layernorm_small":
        return MLPVaRSoftplusSiLULayerNorm(d_in=d_in, hidden=(64, 32), dropout=0.10)
    raise ValueError(f"Arquitectura no soportada: {architecture}")


def train_one_model(X_train, y_train, d_in, seed, w_loss, architecture, alpha=0.99, max_epochs=200, lr=5e-5, batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]
    model = build_model(d_in=d_in, architecture=architecture)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=batch_size, shuffle=False)
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)
    best_loss = np.inf
    best_state = None
    patience_counter = 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## Ejecucion de experimentos

In [7]:
def run_experiment(exp):
    eval_dates = dataset.loc[pd.Timestamp(EVAL_START):pd.Timestamp(EVAL_END)].index
    feature_cols = FEATURE_SETS[exp["feature_set"]]
    var_preds = []
    real_targets = []
    dates = []
    current_model = None
    muX, sdX = None, None

    desc = f"{exp['key']} w=1"
    for i, current_date in enumerate(tqdm(eval_dates, desc=desc, dynamic_ncols=True)):
        if i % RETRAIN_EVERY == 0:
            train_end = current_date - pd.Timedelta(days=1)
            train_df = dataset.loc[:train_end].tail(WINDOW)
            if len(train_df) < 250:
                if current_model is None:
                    continue
            else:
                X_train = train_df[feature_cols].values.astype(np.float32)
                y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)
                muX = X_train.mean(axis=0, keepdims=True)
                sdX = X_train.std(axis=0, keepdims=True) + 1e-8
                X_train = (X_train - muX) / sdX
                current_model = train_one_model(
                    X_train,
                    y_train,
                    d_in=X_train.shape[1],
                    seed=SEED,
                    w_loss=W_LOSS,
                    architecture=exp["architecture"],
                    alpha=ALPHA,
                )

        if current_model is None or muX is None or sdX is None:
            continue

        test_df = dataset.loc[[current_date]]
        X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
        y_test = test_df["target"].values.astype(np.float32).reshape(-1, 1)

        current_model.eval()
        with torch.no_grad():
            pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

        var_preds.append(float(pred))
        real_targets.append(float(y_test.reshape(-1)[0]))
        dates.append(current_date)

    pred_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
    pred_df = pred_df.loc[EVAL_START:EVAL_END].dropna().copy()
    pred_df["viol"] = (pred_df["loss_real"] > pred_df["VaR_pred"]).astype(int)
    pred_df["year"] = pred_df.index.year
    return pred_df


results = {}
summary_parts = []
yearly_parts = []
gap_parts = []

for exp in EXPERIMENTS:
    print(f"\n{'=' * 90}\nEjecutando: {exp['label']}\n{'=' * 90}")
    pred_df = run_experiment(exp)
    results[exp["key"]] = pred_df

    summary = evaluate_var_model(pred_df, model_label=exp["label"], alpha=ALPHA, sig=SIG)
    yearly = evaluate_by_year(pred_df, model_label=exp["label"], alpha=ALPHA)
    violations = violation_table(pred_df)
    gaps = violation_gap_summary(pred_df)
    gaps["Model"] = exp["label"]

    pred_path = DATA_DIR / exp["pred_file"]
    summary_path = DATA_DIR / f"{exp['prefix']}_summary.csv"
    yearly_path = DATA_DIR / f"{exp['prefix']}_yearly.csv"
    violations_path = DATA_DIR / f"{exp['prefix']}_violations.csv"

    pred_df.to_csv(pred_path)
    summary.to_csv(summary_path, index=False)
    yearly.to_csv(yearly_path, index=False)
    violations.to_csv(violations_path)

    print("Guardado:", pred_path)
    print("Guardado:", summary_path)
    print("Guardado:", yearly_path)
    print("Guardado:", violations_path)
    plausibility_report(pred_df)
    display(summary)
    display(yearly)
    display(violations)

    summary_parts.append(summary)
    yearly_parts.append(yearly)
    gap_parts.append(gaps)

new_summary = pd.concat(summary_parts, ignore_index=True)
new_yearly = pd.concat(yearly_parts, ignore_index=True)
gap_summary = pd.DataFrame(gap_parts)


Ejecutando: Softplus + dynamic SiLU LayerNorm dropout 0.20


silu_dropout020 w=1: 100%|██████████| 3762/3762 [16:26<00:00,  3.81it/s]

Guardado: ..\data\nn_softplus_silu_dropout020_2010_2024_w1.csv
Guardado: ..\data\softplus_silu_dropout020_2010_2024_w1_summary.csv
Guardado: ..\data\softplus_silu_dropout020_2010_2024_w1_yearly.csv
Guardado: ..\data\softplus_silu_dropout020_2010_2024_w1_violations.csv

loss_real
min: -0.07361457496881485
max: 0.07843015342950821
mean: -0.00016187175334081243
p95: 0.009858225146308528
p99: 0.017429363690316665
p99.9: 0.038501730956141275

VaR_pred
min: 0.006567837204784155
max: 0.07085435092449188
mean: 0.022171770971107996
p95: 0.03924662731587886
p99: 0.05142480205744504
p99.9: 0.06449041600525406
n_negative_var: 0
max VaR: 2019-09-25 00:00:00 0.07085435092449188
min VaR: 2014-06-12 00:00:00 0.006567837204784155


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus + dynamic SiLU LayerNorm dropout 0.20,1,3762,37.62,43,0.01143,0.00143,0.00143,0.000302,0.022331,0.022172,0.070854,0.006568,0.027371,0.068118,0,0.388732,PASS,0.560315,PASS,0.008035,FAIL,0.319027,NO


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + dynamic SiLU LayerNorm dropout 0.20,2010,252,2.52,3,0.011905,0.001905,0.000279,0.020007,0.019853,0.044664,0.008379,0.026197,-0.000381,0,0.767969,PASS,0.923289,PASS,0.998540,PASS
1,Softplus + dynamic SiLU LayerNorm dropout 0.20,2011,252,2.52,3,0.011905,0.001905,0.000239,0.022397,0.022372,0.047675,0.008454,0.026702,-0.000277,0,0.767969,PASS,0.923289,PASS,0.998540,PASS
2,Softplus + dynamic SiLU LayerNorm dropout 0.20,2012,249,2.49,4,0.016064,0.006064,0.000257,0.019619,0.019495,0.042005,0.008751,0.019407,-0.000117,0,0.376725,PASS,0.633651,PASS,0.008719,FAIL
3,Softplus + dynamic SiLU LayerNorm dropout 0.20,2013,251,2.51,2,0.007968,0.002032,0.000301,0.020725,0.020531,0.050620,0.008500,0.029816,0.000066,0,0.737311,PASS,0.930176,PASS,0.999324,PASS
4,Softplus + dynamic SiLU LayerNorm dropout 0.20,2014,252,2.52,1,0.003968,0.006032,0.000181,0.018114,0.018106,0.051477,0.006568,0.025232,0.000438,0,0.273177,PASS,0.546423,PASS,0.818540,PASS
5,Softplus + dynamic SiLU LayerNorm dropout 0.20,2015,252,2.52,2,0.007937,0.002063,0.000299,0.027463,0.027404,0.059569,0.009766,0.019679,0.000456,0,0.732712,PASS,0.928317,PASS,0.999282,PASS
6,Softplus + dynamic SiLU LayerNorm dropout 0.20,2016,250,2.50,4,0.016000,0.006000,0.000277,0.025924,0.025897,0.064905,0.009147,0.016855,-0.000410,0,0.380484,PASS,0.637706,PASS,0.971186,PASS
7,Softplus + dynamic SiLU LayerNorm dropout 0.20,2017,249,2.49,1,0.004016,0.005984,0.000203,0.019033,0.019016,0.039454,0.008444,0.014269,-0.000486,0,0.280550,PASS,0.556404,PASS,0.831176,PASS
8,Softplus + dynamic SiLU LayerNorm dropout 0.20,2018,250,2.50,4,0.016000,0.006000,0.000282,0.020724,0.020562,0.049268,0.008551,0.018189,0.000311,0,0.380484,PASS,0.637706,PASS,0.004472,FAIL
9,Softplus + dynamic SiLU LayerNorm dropout 0.20,2019,251,2.51,2,0.007968,0.002032,0.000270,0.025426,0.025407,0.070854,0.008844,0.018302,-0.000611,0,0.737311,PASS,0.930176,PASS,0.999324,PASS


...

silu_selected_features w=1: 100%|██████████| 3762/3762 [12:05<00:00,  5.19it/s]


Guardado: ..\data\nn_softplus_silu_selected_features_2010_2024_w1.csv
Guardado: ..\data\softplus_silu_selected_features_2010_2024_w1_summary.csv
Guardado: ..\data\softplus_silu_selected_features_2010_2024_w1_yearly.csv
Guardado: ..\data\softplus_silu_selected_features_2010_2024_w1_violations.csv

loss_real
min: -0.07361457496881485
max: 0.07843015342950821
mean: -0.00016187175334081243
p95: 0.009858225146308528
p99: 0.017429363690316665
p99.9: 0.038501730956141275

VaR_pred
min: 0.010480855591595173
max: 0.044859908521175385
mean: 0.02372109894068879
p95: 0.032830569520592684
p99: 0.03628015786409377
p99.9: 0.043416422575712384
n_negative_var: 0
max VaR: 2020-11-16 00:00:00 0.044859908521175385
min VaR: 2013-07-02 00:00:00 0.010480855591595173


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus + SiLU LayerNorm selected features,1,3762,37.62,20,0.005316,0.004684,-0.004684,0.000308,0.023862,0.023721,0.04486,0.010481,0.02292,0.04486,0,0.001523,FAIL,0.000102,FAIL,0.0,FAIL,0.000542,NO


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,Softplus + SiLU LayerNorm selected features,2010,252,2.52,0,0.000000,0.010000,0.000254,0.025012,0.025012,0.038482,0.012626,0.026197,-0.000381,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
1,Softplus + SiLU LayerNorm selected features,2011,252,2.52,2,0.007937,0.002063,0.000258,0.023945,0.023912,0.039274,0.012433,0.026702,-0.000277,0,0.732712,PASS,0.928317,PASS,9.992823e-01,PASS
2,Softplus + SiLU LayerNorm selected features,2012,249,2.49,0,0.000000,0.010000,0.000252,0.025123,0.025123,0.043327,0.013020,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
3,Softplus + SiLU LayerNorm selected features,2013,251,2.51,2,0.007968,0.002032,0.000274,0.023206,0.023119,0.043115,0.010481,0.029816,0.000066,0,0.737311,PASS,0.930176,PASS,9.993244e-01,PASS
4,Softplus + SiLU LayerNorm selected features,2014,252,2.52,1,0.003968,0.006032,0.000257,0.023322,0.023264,0.041271,0.010712,0.025232,0.000438,0,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS
5,Softplus + SiLU LayerNorm selected features,2015,252,2.52,2,0.007937,0.002063,0.000241,0.021963,0.021910,0.036449,0.010898,0.019679,0.000456,0,0.732712,PASS,0.928317,PASS,9.992823e-01,PASS
6,Softplus + SiLU LayerNorm selected features,2016,250,2.50,0,0.000000,0.010000,0.000245,0.024077,0.024077,0.040522,0.013759,0.016855,-0.000410,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
7,Softplus + SiLU LayerNorm selected features,2017,249,2.49,0,0.000000,0.010000,0.000257,0.025198,0.025198,0.043892,0.013411,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,Softplus + SiLU LayerNorm selected features,2018,250,2.50,1,0.004000,0.006000,0.000231,0.023086,0.023080,0.043846,0.010514,0.018189,0.000311,0,0.278071,PASS,0.553066,PASS,8.270162e-01,PASS
9,Softplus + SiLU LayerNorm selected features,2019,251,2.51,0,0.000000,0.010000,0.000238,0.023180,0.023180,0.039670,0.012743,0.018302,-0.000611,0,NaN,FAIL,NaN,FAIL,NaN,FAIL


,VaR_pred,loss_real,viol,year,exceedance,exceedance_ratio
2011-05-04,0.023072,0.023495,1,2011,0.000424,1.018356
2011-09-21,0.023066,0.026702,1,2011,0.003636,1.157642
2013-04-12,0.020880,0.029816,1,2013,0.008936,1.427951
2013-06-19,0.024615,0.026508,1,2013,0.001892,1.076884
2014-11-26,0.018011,0.025232,1,2014,0.007221,1.400919
2015-04-07,0.012381,0.016571,1,2015,0.004190,1.338434
2015-08-31,0.017223,0.019679,1,2015,0.002455,1.142546
2018-12-21,0.017401,0.018189,1,2018,0.000788,1.045302
2020-03-05,0.016947,0.020594,1,2020,0.003647,1.215204
2020-03-06,0.029234,0.067146,1,2020,0.037912,2.296865


## Comparacion final

In [8]:
def load_prediction_csv(path, var_col="VaR_pred"):
    df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
    df = df.loc[EVAL_START:EVAL_END].copy()
    if var_col != "VaR_pred":
        df = df.rename(columns={var_col: "VaR_pred"})
    if "viol" not in df.columns:
        df["viol"] = (df["loss_real"] > df["VaR_pred"]).astype(int)
    if "year" not in df.columns:
        df["year"] = df.index.year
    return df

comparison_rows = []
gap_rows = []

reference_files = [
    ("Softplus base w=1", DATA_DIR / "nn_softplus_validation_w1.csv", DATA_DIR / "nn_softplus_test_w1.csv"),
    ("Softplus + ratios + shocks", DATA_DIR / "nn_softplus_pruebas_shocks_2010_2024_w1.csv", None),
    ("Softplus + dynamic features + SiLU LayerNorm", DATA_DIR / "nn_softplus_dynamic_silu_layernorm_2010_2024_w1.csv", None),
    ("Softplus + dynamic SiLU LayerNorm + rolling winsor", DATA_DIR / "nn_softplus_silu_winsor_2010_2024_w1.csv", None),
    ("Softplus + SiLU LayerNorm + downside pressure", DATA_DIR / "nn_softplus_silu_downside_pressure_2010_2024_w1.csv", None),
]

for label, path_a, path_b in reference_files:
    if not path_a.exists():
        continue
    if path_b is not None and path_b.exists():
        ref_df = pd.concat([load_prediction_csv(path_a), load_prediction_csv(path_b)]).sort_index()
        ref_df = ref_df[~ref_df.index.duplicated(keep="last")].copy()
        ref_df = ref_df.loc[EVAL_START:EVAL_END]
    else:
        ref_df = load_prediction_csv(path_a)
    comparison_rows.append(evaluate_var_model(ref_df, model_label=label, alpha=ALPHA, sig=SIG))
    gaps = violation_gap_summary(ref_df)
    gaps["Model"] = label
    gap_rows.append(gaps)

comparison_rows.append(new_summary)
gap_rows.extend(gap_parts)

comparison = pd.concat(comparison_rows, ignore_index=True)
comparison_path = DATA_DIR / f"{EXPERIMENT_PREFIX}_comparison.csv"
comparison.to_csv(comparison_path, index=False)

gap_comparison = pd.DataFrame(gap_rows)
gap_path = DATA_DIR / f"{EXPERIMENT_PREFIX}_gap_comparison.csv"
gap_comparison.to_csv(gap_path, index=False)

print("Guardado:", comparison_path)
print("Guardado:", gap_path)
print("\nConfirmacion metodologica: la funcion var_loss es la misma pinball que en los experimentos Softplus previos.")

cols = [
    "Model", "w", "T", "Expected Viol.", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
    "Tick Loss", "Winkler Score", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
    "Mean VaR 2020", "Max VaR 2020", "n_negative_var", "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test", "p_Mean", "Valid(CC&DQ)",
]

display(comparison[cols].sort_values(["DQ Test", "CC Test", "UC Test", "VR Gap", "Tick Loss"], ascending=[False, False, False, True, True]))

print("\nResumen de gaps entre violaciones")
display(gap_comparison[["Model", "violations", "median_gap_days", "gaps_le_5", "gaps_le_20"]])

print("\nComparacion anual de los nuevos experimentos")
display(new_yearly.sort_values(["Model", "year"]))

print("\nAnos problematicos de los nuevos experimentos")
display(new_yearly.sort_values(["Violation Rate", "Violations"], ascending=False).head(20))

Guardado: ..\data\softplus_silu_regularized_2010_2024_w1_comparison.csv
Guardado: ..\data\softplus_silu_regularized_2010_2024_w1_gap_comparison.csv

Confirmacion metodologica: la funcion var_loss es la misma pinball que en los experimentos Softplus previos.


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Mean VaR 2020,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
4,Softplus + SiLU LayerNorm + downside pressure,1,3762,37.62,39,0.010367,0.000367,0.000367,0.000294,0.020153,0.019966,0.052294,0.007711,0.019653,0.040079,0,0.822151,PASS,0.183812,PASS,1.268545e-10,FAIL,0.335321,NO
3,Softplus + dynamic SiLU LayerNorm + rolling wi...,1,3762,37.62,41,0.010898,0.000898,0.000898,0.000303,0.022349,0.022188,0.070843,0.006599,0.027348,0.067945,0,0.585114,PASS,0.664387,PASS,3.974594e-03,FAIL,0.417825,NO
2,Softplus + dynamic features + SiLU LayerNorm,1,3762,37.62,42,0.011164,0.001164,0.001164,0.000302,0.022348,0.022189,0.070849,0.006577,0.027367,0.067910,0,0.481090,PASS,0.618056,PASS,5.803898e-03,FAIL,0.368316,NO
5,Softplus + dynamic SiLU LayerNorm dropout 0.20,1,3762,37.62,43,0.011430,0.001430,0.001430,0.000302,0.022331,0.022172,0.070854,0.006568,0.027371,0.068118,0,0.388732,PASS,0.560315,PASS,8.034862e-03,FAIL,0.319027,NO
1,Softplus + ratios + shocks,1,3762,37.62,32,0.008506,0.001494,-0.001494,0.000274,0.018858,0.018684,0.028376,0.008985,0.019036,0.028376,0,0.344590,PASS,0.060605,PASS,0.000000e+00,FAIL,0.135065,NO
0,Softplus base w=1,1,3762,37.62,36,0.009569,0.000431,-0.000431,0.000283,0.018417,0.018217,0.030089,0.007403,0.017326,0.030089,0,0.789181,PASS,0.016972,FAIL,0.000000e+00,FAIL,0.268718,NO
7,Softplus + SiLU LayerNorm selected features,1,3762,37.62,20,0.005316,0.004684,-0.004684,0.000308,0.023862,0.023721,0.044860,0.010481,0.022920,0.044860,0,0.001523,FAIL,0.000102,FAIL,0.000000e+00,FAIL,0.000542,NO
6,Softplus + dynamic SiLU LayerNorm small,1,3762,37.62,57,0.015152,0.005152,0.005152,0.000304,0.017921,0.017667,0.043033,0.008186,0.017277,0.036606,0,0.003165,FAIL,0.000090,FAIL,0.000000e+00,FAIL,0.001085,NO



Resumen de gaps entre violaciones


,Model,violations,median_gap_days,gaps_le_5,gaps_le_20
0,Softplus base w=1,36,35.0,8,16
1,Softplus + ratios + shocks,32,53.0,8,12
2,Softplus + dynamic features + SiLU LayerNorm,42,68.0,5,11
3,Softplus + dynamic SiLU LayerNorm + rolling wi...,41,77.5,5,11
4,Softplus + SiLU LayerNorm + downside pressure,39,59.5,6,9
5,Softplus + dynamic SiLU LayerNorm dropout 0.20,43,67.0,5,11
6,Softplus + dynamic SiLU LayerNorm small,57,27.0,14,26
7,Softplus + SiLU LayerNorm selected features,20,68.0,6,9



Comparacion anual de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
30,Softplus + SiLU LayerNorm selected features,2010,252,2.52,0,0.000000,0.010000,0.000254,0.025012,0.025012,0.038482,0.012626,0.026197,-0.000381,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
31,Softplus + SiLU LayerNorm selected features,2011,252,2.52,2,0.007937,0.002063,0.000258,0.023945,0.023912,0.039274,0.012433,0.026702,-0.000277,0,7.327118e-01,PASS,9.283165e-01,PASS,9.992823e-01,PASS
32,Softplus + SiLU LayerNorm selected features,2012,249,2.49,0,0.000000,0.010000,0.000252,0.025123,0.025123,0.043327,0.013020,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
33,Softplus + SiLU LayerNorm selected features,2013,251,2.51,2,0.007968,0.002032,0.000274,0.023206,0.023119,0.043115,0.010481,0.029816,0.000066,0,7.373115e-01,PASS,9.301764e-01,PASS,9.993244e-01,PASS
34,Softplus + SiLU LayerNorm selected features,2014,252,2.52,1,0.003968,0.006032,0.000257,0.023322,0.023264,0.041271,0.010712,0.025232,0.000438,0,2.731770e-01,PASS,5.464228e-01,PASS,8.185397e-01,PASS
35,Softplus + SiLU LayerNorm selected features,2015,252,2.52,2,0.007937,0.002063,0.000241,0.021963,0.021910,0.036449,0.010898,0.019679,0.000456,0,7.327118e-01,PASS,9.283165e-01,PASS,9.992823e-01,PASS
36,Softplus + SiLU LayerNorm selected features,2016,250,2.50,0,0.000000,0.010000,0.000245,0.024077,0.024077,0.040522,0.013759,0.016855,-0.000410,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
37,Softplus + SiLU LayerNorm selected features,2017,249,2.49,0,0.000000,0.010000,0.000257,0.025198,0.025198,0.043892,0.013411,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
38,Softplus + SiLU LayerNorm selected features,2018,250,2.50,1,0.004000,0.006000,0.000231,0.023086,0.023080,0.043846,0.010514,0.018189,0.000311,0,2.780715e-01,PASS,5.530661e-01,PASS,8.270162e-01,PASS
39,Softplus + SiLU LayerNorm selected features,2019,251,2.51,0,0.000000,0.010000,0.000238,0.023180,0.023180,0.039670,0.012743,0.018302,-0.000611,0,NaN,FAIL,NaN,FAIL,NaN,FAIL



Anos problematicos de los nuevos experimentos


,Model,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
25,Softplus + dynamic SiLU LayerNorm small,2020,251,2.51,15,0.059761,0.049761,0.001319,0.019577,0.017277,0.036606,0.010024,0.078430,-0.000757,0,6.226481e-08,FAIL,2.421541e-07,FAIL,4.100440e-07,FAIL
27,Softplus + dynamic SiLU LayerNorm small,2022,251,2.51,11,0.043825,0.033825,0.000316,0.017843,0.017551,0.043033,0.008911,0.026797,0.000322,0,6.962880e-05,FAIL,7.736408e-05,FAIL,8.889256e-04,FAIL
40,Softplus + SiLU LayerNorm selected features,2020,251,2.51,10,0.039841,0.029841,0.001127,0.024719,0.022920,0.044860,0.011090,0.078430,-0.000757,0,3.296171e-04,FAIL,2.353750e-04,FAIL,1.968425e-13,FAIL
10,Softplus + dynamic SiLU LayerNorm dropout 0.20,2020,251,2.51,8,0.031873,0.021873,0.000955,0.028733,0.027371,0.068118,0.008639,0.078430,-0.000757,0,5.556650e-03,FAIL,1.069465e-02,FAIL,9.923955e-02,PASS
16,Softplus + dynamic SiLU LayerNorm small,2011,252,2.52,7,0.027778,0.017778,0.000335,0.017968,0.017654,0.035508,0.009688,0.026702,-0.000277,0,1.986121e-02,FAIL,2.273129e-02,FAIL,2.931819e-06,FAIL
23,Softplus + dynamic SiLU LayerNorm small,2018,250,2.50,5,0.020000,0.010000,0.000228,0.016882,0.016754,0.031053,0.008866,0.018189,0.000311,0,1.618549e-01,PASS,3.392998e-01,PASS,5.986945e-03,FAIL
12,Softplus + dynamic SiLU LayerNorm dropout 0.20,2022,251,2.51,5,0.019920,0.009920,0.000304,0.024627,0.024501,0.050860,0.009109,0.026797,0.000322,0,1.640397e-01,PASS,3.428915e-01,PASS,8.813249e-01,PASS
2,Softplus + dynamic SiLU LayerNorm dropout 0.20,2012,249,2.49,4,0.016064,0.006064,0.000257,0.019619,0.019495,0.042005,0.008751,0.019407,-0.000117,0,3.767250e-01,PASS,6.336506e-01,PASS,8.719131e-03,FAIL
6,Softplus + dynamic SiLU LayerNorm dropout 0.20,2016,250,2.50,4,0.016000,0.006000,0.000277,0.025924,0.025897,0.064905,0.009147,0.016855,-0.000410,0,3.804837e-01,PASS,6.377058e-01,PASS,9.711856e-01,PASS
8,Softplus + dynamic SiLU LayerNorm dropout 0.20,2018,250,2.50,4,0.016000,0.006000,0.000282,0.020724,0.020562,0.049268,0.008551,0.018189,0.000311,0,3.804837e-01,PASS,6.377058e-01,PASS,4.472390e-03,FAIL


## Lectura esperada

Un experimento solo es candidato si mejora el fallo del modelo 2 sin destruir la cobertura global:

- objetivo principal: `UC`, `CC` y `DQ` globales en `PASS`;
- si `DQ` no pasa, mirar si sube mucho `p_DQ` frente al modelo 2;
- la tasa de violacion deberia quedar cerca del 1%;
- el VaR medio no debe subir de forma generalizada;
- marzo de 2020 y 2022 son los anos clave para diagnosticar clustering.